# PNCP public procurement: how the raw layer is obtained

**Publisher:** Portal Nacional de Contratações Públicas (PNCP)  
**Official source:** <https://pncp.gov.br/api/consulta/v1>

Procurement notices (including editais), contracts, price-registration records (atas) and annual procurement plans (PCA) with their items, published through the PNCP consultation API.

**Grain and keys:** One row per natural key, keeping the latest version by `dataAtualizacaoGlobal`: `numero_controle_pncp` for notices and contracts, `numero_controle_pncp_ata` for atas, `id_pca_pncp` for plans. Municipality is reachable through `unidade_codigo_ibge` (IBGE code).

**Released as:**

- `pncp-analytics`: semantic (6 tables)
- `pncp-raw-trusted-part-1`: raw, trusted (40 tables)
- `pncp-raw-trusted-part-2`: trusted (1 tables)

Every table and column is documented in the [interactive data map](https://lucas.rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

Executed against the live source when this notebook was written; the outputs below are from that run.

In [1]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. Fetch one page of procurement notices

The consultation API needs no key. Notices (`contratacoes`) require the modality code and a page size of at most 50. The `/atualizacao` axis returns everything that entered or changed in the window, which is how the released tables stay current.

In [2]:
BASE = "https://pncp.gov.br/api/consulta/v1"
params = {"dataInicial": "20260901", "dataFinal": "20260902", "codigoModalidadeContratacao": 8, "pagina": 1, "tamanhoPagina": 10}
page = get(f"{BASE}/contratacoes/atualizacao", params=params).json()
print({k: v for k, v in page.items() if k != "data"})
print(len(page["data"]), "notices on this page")
notice = page["data"][0]
print(json.dumps({k: notice[k] for k in list(notice)[:12]}, indent=1, ensure_ascii=False, default=str)[:1500])

{'totalRegistros': 6352, 'totalPaginas': 636, 'numeroPagina': 1, 'paginasRestantes': 635, 'empty': False}
10 notices on this page
{
 "valorTotalHomologado": 53518.4,
 "orgaoEntidade": {
  "cnpj": "75924290000169",
  "razaoSocial": "MUNICIPIO DE PEROLA D'OESTE",
  "poderId": "N",
  "esferaId": "M"
 },
 "dataInclusao": "2026-08-25T00:00:14",
 "dataAtualizacao": "2026-08-25T00:00:14",
 "linkSistemaOrigem": null,
 "linkProcessoEletronico": null,
 "justificativaPresencial": null,
 "objetoCompra": "Contratação de empresa especializada para prestação de serviços de oficinas de dança no âmbito do Serviço de Convivência e Fortalecimento de Vínculos (SCFV), vinculado à Secre-taria Municipal de Assistência Social, destinadas a crianças e adolescentes, incluindo plane-jamento das atividades, elaboração de relatórios e desenvolvimento de ações voltadas ao fortalecimento dos vínculos familiares e comunitários, no Município de Pérola D’Oeste/PR",
 "informacaoComplementar": "conforme edital de dispens

## 2. The other three domains

Same pattern, different path and key. Contracts and price-registration records accept a page size of 500. The annual plan endpoint (PCA) uses `dataInicio` and `dataFim`.

In [3]:
for domain, extra in [("contratos", {}), ("atas", {}), ("pca", {})]:
    p = {"dataInicial": "20260901", "dataFinal": "20260901", "pagina": 1, "tamanhoPagina": 10}
    if domain == "pca":
        p = {"dataInicio": "20260901", "dataFim": "20260901", "pagina": 1, "tamanhoPagina": 10}
    resp = get(f"{BASE}/{domain}/atualizacao", params=p)
    body = resp.json() if resp.status_code == 200 and resp.content else {}
    print(domain, resp.status_code, "records:", len(body.get("data", [])), "| total pages:", body.get("totalPaginas"))

contratos 200 records: 10 | total pages: 971


atas 200 records: 10 | total pages: 293


pca 204 records: 0 | total pages: None


## 3. From the JSON to the trusted columns

The raw layer stores the record untouched (`payload_json`) plus control columns. The trusted layer flattens nested objects into snake_case columns: for example `orgaoEntidade.cnpj` becomes `orgao_cnpj` and `unidadeOrgao.codigoIbge` becomes `unidade_codigo_ibge`, the key that joins a notice to a municipality. The first columns of the released `pncp_contratacoes` table:

`numero_controle_pncp`, `ano_compra`, `sequencial_compra`, `numero_compra`, `processo`, `modalidade_id`, `modalidade_nome`, `modo_disputa_id`, `modo_disputa_nome`, `tipo_instrumento_convocatorio_codigo`, `tipo_instrumento_convocatorio_nome`, `situacao_compra_id`, `situacao_compra_nome`, `srp`, `objeto_compra`, `informacao_complementar`, `justificativa_presencial`, `valor_total_estimado`, `valor_total_homologado`, `data_inclusao`, `data_publicacao_pncp`, `data_atualizacao`, `data_abertura_proposta`, `data_encerramento_proposta`, `data_atualizacao_global`, `amparo_legal_codigo`, `amparo_legal_nome`, `amparo_legal_descricao`, `orgao_cnpj`, `orgao_razao_social`, `orgao_poder_id`, `orgao_esfera_id`, `unidade_uf_sigla`, `unidade_uf_nome`, `unidade_codigo`, `unidade_nome`, `unidade_municipio_nome`, `unidade_codigo_ibge`, `link_sistema_origem`, `link_processo_eletronico`

In [4]:
def flatten(obj, prefix=""):
    out = {}
    for key, value in obj.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            out.update(flatten(value, name + "."))
        else:
            out[name] = value
    return out

import pandas as pd
pd.DataFrame([flatten(n) for n in page["data"]]).iloc[:, :14]

,valorTotalHomologado,orgaoEntidade.cnpj,orgaoEntidade.razaoSocial,orgaoEntidade.poderId,orgaoEntidade.esferaId,dataInclusao,dataAtualizacao,linkSistemaOrigem,linkProcessoEletronico,justificativaPresencial,objetoCompra,informacaoComplementar,dataAberturaProposta,dataEncerramentoProposta
0,53518.40,75924290000169,MUNICIPIO DE PEROLA D'OESTE,N,M,2026-08-25T00:00:14,2026-08-25T00:00:14,None,None,None,Contratação de empresa especializada para pres...,conforme edital de dispensa de licitação,2026-08-21T08:00:01,2026-08-21T15:00:01
1,13900.00,80881915000192,MUNICIPIO DE LINDOESTE,N,M,2026-08-28T00:00:29,2026-08-28T00:00:29,None,None,CONTRATAÇÃO POR JUSTIFICATIVA DE PREÇO,CONTRATAÇÃO DE PESSOA JURIDICA ESPECIALIZADA P...,None,None,None
2,19977.00,07854402000100,EMPRESA MUNICIPAL DE AGUA E SANEAMENTO DE BALN...,E,M,2026-09-01T00:00:03,2026-09-01T00:00:03,None,None,None,AQUISIÇÃO DE PORTAS DE VIDRO QUE SERÃO NSTALAD...,None,None,None
3,61400.00,40637159000136,FUNDACAO HOSPITALAR DE FEIRA DE SANTANA,E,M,2026-08-27T10:10:52,2026-08-27T10:10:52,None,None,None,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA FORN...,,None,None
4,28060.48,94436474000124,MUNICIPIO DE XANGRI-LA,N,M,2026-08-31T17:32:56,2026-09-01T00:10:02,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Pedido gerado a partir do resultado Solicitaçã...,None,None,None
5,1640.00,97229181000164,MUNICIPIO DE SAO SEPE,N,M,2026-09-01T00:10:03,2026-09-01T00:10:03,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Aquisição de materiais e serviço de mão de obr...,None,None,None
6,2306.20,97229181000164,MUNICIPIO DE SAO SEPE,N,M,2026-08-31T14:00:08,2026-09-01T00:10:09,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Aquisição de peças para o conserto do veículo ...,None,None,None
7,6680.00,90898487000164,MUNICIPIO DE COTIPORA,N,M,2026-08-31T13:48:28,2026-09-01T00:10:09,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Contratação de empresa especializada para forn...,None,None,None
8,711.00,90898487000164,MUNICIPIO DE COTIPORA,N,M,2026-08-31T13:24:15,2026-09-01T00:10:11,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Contratação de empresa especializada para manu...,None,None,None
9,2694.20,97229181000164,MUNICIPIO DE SAO SEPE,N,M,2026-08-31T13:34:25,2026-09-01T00:10:11,https://portal.tce.rs.gov.br/aplicprod/f?p=505...,None,None,Aquisição de peças para manutenção do veículo ...,None,None,None


## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://lucas.rangeltech.net/datamap/#/subject/pncp).

In [5]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/pncp-raw-trusted-part-1", path="trusted__pncp_dom_tipos_parte_envolvida.parquet")
pd.read_parquet(path).head()

C:\Users\lucas.rangel\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/0.99k [00:00<?, ?B/s]

100%|██████████| 0.99k/0.99k [00:00<?, ?B/s]

,descricao,nome,id
0,None,Gerenciadora,1
1,None,Participante,2
2,None,Não Participante,3
